In [1]:
!pip install -q wikipedia-api langchain sentence-transformers faiss-cpu rank_bm25 transformers accelerate bitsandbytes gradio


In [3]:
import wikipediaapi
import json
import random
import os
import time
import requests

# 1. SET A UNIQUE USER AGENT (Change the email to anything)
USER_AGENT = "MyProjectBot/1.0 (contact_your_email@example.com)"
wiki_wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language='en')

def get_random_titles(count):
    """Gets a list of random Wikipedia titles in one API call."""
    titles = []
    # Wikipedia API allows up to 500 random titles at once
    url = f"https://en.wikipedia.org/w/api.php?action=query&list=random&rnlimit={count}&rnnamespace=0&format=json"
    headers = {'User-Agent': USER_AGENT}

    response = requests.get(url, headers=headers).json()
    for page in response['query']['random']:
        titles.append(page['title'])
    return titles

def fetch_page_data(titles, target_count, existing_urls):
    """Fetches full content for a list of titles and validates length."""
    valid_pages = []
    for title in titles:
        if len(valid_pages) >= target_count:
            break

        try:
            page = wiki_wiki.page(title)
            # Check length (minimum 200 words)
            if page.exists() and len(page.text.split()) >= 200 and page.fullurl not in existing_urls:
                valid_pages.append({
                    "title": page.title,
                    "url": page.fullurl,
                    "text": page.text
                })
                existing_urls.add(page.fullurl)
                # Small sleep to be polite to the API
                time.sleep(0.1)
        except Exception as e:
            continue
    return valid_pages

def get_valid_wiki_pages(target_count, existing_urls=None):
    if existing_urls is None:
        existing_urls = set()

    final_pages = []
    while len(final_pages) < target_count:
        needed = target_count - len(final_pages)
        print(f"Searching for {needed} more valid pages...")

        # Get a batch of random titles (fetch more than needed because some will be < 200 words)
        candidate_titles = get_random_titles(min(needed * 3, 500))
        new_pages = fetch_page_data(candidate_titles, needed, existing_urls)
        final_pages.extend(new_pages)

    return final_pages

# --- Execution ---

# 1. Handle Fixed URLs (200)
FIXED_FILE = "fixed_urls.json"
if os.path.exists(FIXED_FILE):
    with open(FIXED_FILE, 'r') as f:
        fixed_data = json.load(f)
    print(f"Loaded {len(fixed_data)} fixed URLs.")
else:
    print("Sampling 200 fixed URLs...")
    fixed_data = get_valid_wiki_pages(200)
    with open(FIXED_FILE, 'w') as f:
        json.dump(fixed_data, f)

# 2. Handle Random URLs (300)
print("Sampling 300 random URLs...")
random_data = get_valid_wiki_pages(300, existing_urls={p['url'] for p in fixed_data})

total_corpus = fixed_data + random_data
print(f"Success! Total Corpus Size: {len(total_corpus)} articles.")


Sampling 200 fixed URLs...
Searching for 200 more valid pages...
Sampling 300 random URLs...
Searching for 300 more valid pages...
Searching for 31 more valid pages...
Success! Total Corpus Size: 500 articles.


In [4]:
!pip install -q langchain-text-splitters


In [5]:
# Change this:
# from langchain.text_splitter import RecursiveCharacterTextSplitter

# To this:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    length_function=len,
)

all_chunks = []
for doc in total_corpus:
    chunks = text_splitter.split_text(doc['text'])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "id": f"{doc['title']}_{i}",
            "text": chunk,
            "metadata": {"url": doc['url'], "title": doc['title']}
        })

print(f"Total chunks created: {len(all_chunks)}")


Total chunks created: 11121


In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from rank_bm25 import BM25Okapi

# --- 1. Dense Setup ---
# Use a DIFFERENT variable name to avoid conflict with T5 model
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding chunks (this may take a few minutes)...")
chunk_texts = [c['text'] for c in all_chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    batch_size=64  # Speeds up encoding on GPU
)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
print(f"FAISS index built with {index.ntotal} vectors.")

# --- 2. Sparse Setup ---
print("Building BM25 index...")
tokenized_corpus = [doc.lower().split() for doc in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built.")

# --- 3. RRF Implementation ---
def rrf(dense_indices, sparse_indices, k=60, top_n=5):
    """
    Reciprocal Rank Fusion
    RRF_score(d) = sum(1 / (k + rank_i(d)))
    """
    scores = {}

    # Process Dense results
    for rank, idx in enumerate(dense_indices):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    # Process Sparse results
    for rank, idx in enumerate(sparse_indices):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    # Sort by RRF score descending
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:top_n]

# --- 4. Hybrid Search Function ---
def hybrid_search(query, top_k=10, top_n=5):
    # Dense Search using embedding_model (NOT model)
    query_vec = embedding_model.encode([query])  # Uses embedding_model
    _, dense_indices = index.search(
        np.array(query_vec).astype('float32'), top_k
    )

    # Sparse BM25 Search
    tokenized_query = query.lower().split()
    sparse_scores = bm25.get_scores(tokenized_query)
    sparse_indices = np.argsort(sparse_scores)[-top_k:][::-1]

    # RRF Fusion
    final_results = rrf(dense_indices[0].tolist(), sparse_indices.tolist(), top_n=top_n)

    # Return chunks with scores
    return [(all_chunks[idx], score) for idx, score in final_results]

# --- Test ---
results = hybrid_search("What is machine learning?")
print("\nTop Results:")
for chunk, score in results:
    print(f"  Score: {score:.4f} | Title: {chunk['metadata']['title']}")
    print(f"  Preview: {chunk['text'][:100]}...")
    print()


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding chunks (this may take a few minutes)...


Batches:   0%|          | 0/174 [00:00<?, ?it/s]

FAISS index built with 11121 vectors.
Building BM25 index...
BM25 index built.

Top Results:
  Score: 0.0164 | Title: Physical symbol system
  Preview: Connectionism and deep learning
In 2012 AlexNet, a deep learning network, outperformed all other pro...

  Score: 0.0164 | Title: BRP Hilario Ruiz
  Preview: The ship originally designed to carry one bow Mk.3 40 mm gun, one 81 mm  mortar aft, and four 12.7 m...

  Score: 0.0161 | Title: Physical symbol system
  Preview: Artificial intelligence research has succeeded in developing many programs that are capable of intel...

  Score: 0.0161 | Title: Physical symbol system
  Preview: The PSSH refers to "intelligent action" -- that is, the behavior of the machine -- it does not refer...

  Score: 0.0159 | Title: Physical symbol system
  Preview: McCarthy, John; Minsky, Marvin; Rochester, Nathan; Shannon, Claude (1955), A Proposal for the Dartmo...



In [7]:
!pip install -q sentencepiece


In [8]:
# Install required packages
!pip install -q transformers accelerate bitsandbytes

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# ── 4-bit Quantization Config (Saves GPU Memory) ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # Load in 4-bit to save memory
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_id = "HuggingFaceH4/zephyr-7b-beta"

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Loading model {model_id} in 4-bit...")
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"                        # Automatically use GPU
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model loaded on: {device}")
print("Zephyr-7B ready!")


def generate_answer(question, context_chunks):
    """
    Generate detailed answers using Zephyr-7B.
    Zephyr uses a chat format with system/user/assistant roles.
    """
    # Build rich context from retrieved chunks
    context_text = "\n\n".join([
        f"[Source {i+1}] {c['metadata']['title']}:\n{c['text'][:600]}"
        for i, c in enumerate(context_chunks[:5])
    ])

    # Zephyr uses chat template format
    messages = [
        {
            "role": "system",
            "content": """You are a knowledgeable assistant that answers questions
based on the provided sources. Always give detailed, accurate and complete answers.
Use specific facts, names, and dates from the sources.
Never say you don't know - always use the sources to give the best answer."""
        },
        {
            "role": "user",
            "content": f"""Using the sources below, answer the question in detail.

Sources:
{context_text}

Question: {question}

Please provide a detailed and complete answer:"""
        }
    ]

    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=2048,
        truncation=True
    ).to(device)

    # Generate answer
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=400,           # Long detailed answers
            min_new_tokens=100,           # Force minimum length
            do_sample=True,               # Enable sampling
            temperature=0.7,              # Natural responses
            top_p=0.95,                   # Nucleus sampling
            repetition_penalty=1.15,      # Avoid repetition
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer     = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()



Loading tokenizer for HuggingFaceH4/zephyr-7b-beta...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded on: cuda
Zephyr-7B ready!


In [9]:
test_query = "What is the capital of India ?"
retrieved_results = hybrid_search(test_query)
context_chunks = [res[0] for res in retrieved_results]
answer = generate_answer(test_query, context_chunks)
print(f"Query  : {test_query}")
print(f"Answer : {answer}")


Query  : What is the capital of India ?
Answer : The current capital city of India is New Delhi. However, New Delhi is not one of the states or territories that make up India, but rather a territory administered by the central government. The national capital territory of Delhi has its own legislative assembly and government, but ultimate authority lies with the federal government. Before independence in 1947, the colonial power, Britain, had established New Delhi as the seat of government due to its central location between Calcutta (now Kolkata), Mumbai (then Bombay), and Chennai (then Madras). Following India's independence, New Delhi became the permanent national capital, replacing both Calcutta and Mumbai, which had previously served as temporary capitals during different points in history.

In terms of sources, there is no direct mention of the Indian capital in any of the provided sources. But Source 5 does offer some historical context about the naming of a college in honor of 

In [10]:
# ── Cell 6: Load Flan-T5 for QA Generation ───────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

print("Loading Flan-T5 for QA pair generation...")
print("=" * 50)

t5_model_id  = "google/flan-t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_id)

t5_model     = AutoModelForSeq2SeqLM.from_pretrained(
    t5_model_id,
    torch_dtype=torch.float16
)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t5_model = t5_model.to(device)

print(f"✅ Model        : {t5_model_id}")
print(f"✅ Device       : {device}")
print(f"✅ Flan-T5 ready for QA generation!")


Loading Flan-T5 for QA pair generation...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Model        : google/flan-t5-base
✅ Device       : cuda
✅ Flan-T5 ready for QA generation!


In [11]:
import random
import json

def generate_question_from_chunk(chunk_text, title):
    """
    Generate question using whichever model is currently loaded.
    Works with Zephyr, Phi-2, TinyLlama or Flan-T5.
    """
    prompt = f"""Read this text about "{title}" and write ONE
simple factual question starting with Who, What, When, Where or How.
Only write the question, nothing else.

Text: {chunk_text[:300]}

Question:"""

    inputs = tokenizer(          # ← Uses your current model
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = llm_model.generate(   # ← Uses your current model
            **inputs,
            max_new_tokens=30,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    question   = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # Validate
    if len(question) < 5:
        question = f"What is {title} known for?"
    if "?" not in question:
        question = question + "?"

    return question


def generate_answer_from_chunk(question, chunk_text):
    """
    Generate ground truth answer using whichever model is loaded.
    Works with Zephyr, Phi-2, TinyLlama or Flan-T5.
    """
    prompt = f"""Read the text and answer the question clearly
in one or two sentences using only the information in the text.

Text: {chunk_text[:400]}

Question: {question}

Answer:"""

    inputs = tokenizer(          # ← Uses your current model
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = llm_model.generate(   # ← Uses your current model
            **inputs,
            max_new_tokens=80,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer     = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer


# ── Generate 100 QA Pairs ─────────────────────────────────────
print("Generating 100 Q&A pairs...")
print("This will take a few minutes...")
print("=" * 50)

qa_pairs     = []
failed_count = 0

# Only use chunks with enough text
valid_chunks = [
    c for c in all_chunks
    if len(c['text'].split()) > 50
]

print(f"Valid chunks available : {len(valid_chunks)}")

sampled_chunks = random.sample(
    valid_chunks,
    min(120, len(valid_chunks))
)

for i, chunk in enumerate(sampled_chunks):

    if len(qa_pairs) >= 100:
        break

    try:
        title    = chunk['metadata']['title']
        chunk_id = chunk['id']

        # Generate question
        question = generate_question_from_chunk(
            chunk['text'],
            title
        )

        # Validate question
        if len(question.split()) < 4:
            failed_count += 1
            continue

        # Generate answer
        ground_truth = generate_answer_from_chunk(
            question,
            chunk['text']
        )

        # Validate answer
        if len(ground_truth.split()) < 3:
            failed_count += 1
            continue

        qa_pairs.append({
            "id"              : len(qa_pairs) + 1,
            "question"        : question,
            "ground_truth"    : ground_truth,
            "source_url"      : chunk['metadata']['url'],
            "source_title"    : title,
            "source_chunk_id" : chunk_id,
            "category"        : "factual"
        })

        if len(qa_pairs) % 10 == 0:
            print(f"✅ Generated {len(qa_pairs)}/100 Q&A pairs...")

    except Exception as e:
        print(f"⚠️ Error at chunk {i} : {e}")
        failed_count += 1
        continue

# Save
with open("qa_pairs.json", "w") as f:
    json.dump(qa_pairs, f, indent=2)

# Summary
print(f"\n{'='*50}")
print(f"✅ Generated  : {len(qa_pairs)} Q&A pairs")
print(f"❌ Failed     : {failed_count}")
print(f"{'='*50}")

# Show samples
print("\n📋 Sample Q&A Pairs:")
for i in range(min(5, len(qa_pairs))):
    print(f"\n  [{i+1}] Source : {qa_pairs[i]['source_title']}")
    print(f"       Q      : {qa_pairs[i]['question']}")
    print(f"       A      : {qa_pairs[i]['ground_truth']}")
    print(f"       {'-'*40}")


Generating 100 Q&A pairs...
This will take a few minutes...
Valid chunks available : 5034
✅ Generated 10/100 Q&A pairs...
✅ Generated 20/100 Q&A pairs...
✅ Generated 30/100 Q&A pairs...
✅ Generated 40/100 Q&A pairs...
✅ Generated 50/100 Q&A pairs...
✅ Generated 60/100 Q&A pairs...
✅ Generated 70/100 Q&A pairs...

✅ Generated  : 72 Q&A pairs
❌ Failed     : 48

📋 Sample Q&A Pairs:

  [1] Source : Joe Diffie
       Q      : Who recorded a cover of Charlie Rich's song at the end of 1998?
       A      : Diffie recorded a cover of Charlie Rich's song "Behind Closed Doors" at the end of 1998 for the multiple-artist album A Tribute to Tradition on Columbia Records.
       ----------------------------------------

  [2] Source : Scott Cassell
       Q      : Who attempted to set a new world record for longest distance traveled by a diver on September 17, 2011?
       A      : Jeff Cassell attempted to set a new world record for longest distance traveled by a diver on September 17, 2011.
      

In [12]:
def calculate_mrr(qa_pairs, top_k=10):
    """
    Calculate Mean Reciprocal Rank at URL level.
    MRR = (1/|Q|) * sum(1/rank_i)
    """
    reciprocal_ranks = []
    results_log = []

    print("Calculating MRR...")
    for qa in qa_pairs:
        question = qa['question']
        correct_url = qa['source_url']

        # Run hybrid search
        retrieved = hybrid_search(question, top_k=top_k)

        # Find rank of correct URL
        rank = None
        for i, (chunk, score) in enumerate(retrieved):
            if chunk['metadata']['url'] == correct_url:
                rank = i + 1  # Rank starts at 1
                break

        # Calculate reciprocal rank
        rr = 1 / rank if rank else 0
        reciprocal_ranks.append(rr)

        results_log.append({
            "id": qa['id'],
            "question": question,
            "correct_url": correct_url,
            "rank_found": rank if rank else "Not Found",
            "reciprocal_rank": round(rr, 4)
        })

    # Final MRR
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    print(f"\nMRR Calculation Complete!")
    print(f"Total Questions : {len(qa_pairs)}")
    print(f"Found in Results: {sum(1 for r in results_log if r['rank_found'] != 'Not Found')}")
    print(f"MRR Score       : {mrr:.4f}")

    return mrr, results_log

# Run MRR
mrr_score, mrr_log = calculate_mrr(qa_pairs)


Calculating MRR...

MRR Calculation Complete!
Total Questions : 72
Found in Results: 72
MRR Score       : 0.9861


In [13]:
!pip install -q rouge-score

from rouge_score import rouge_scorer

def calculate_rouge(qa_pairs):
    """
    ROUGE-L score between generated answer and ground truth.
    Measures overlap of word sequences.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge1_scores = []
    rougeL_scores = []
    rouge_log = []

    print("Calculating ROUGE scores...")
    for qa in qa_pairs:
        # Get generated answer from RAG system
        retrieved = hybrid_search(qa['question'], top_k=10)
        context_chunks = [res[0] for res in retrieved]
        generated_answer = generate_answer(qa['question'], context_chunks)

        # Score against ground truth
        scores = scorer.score(qa['ground_truth'], generated_answer)

        rouge1_scores.append(scores['rouge1'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

        rouge_log.append({
            "id": qa['id'],
            "question": qa['question'],
            "ground_truth": qa['ground_truth'],
            "generated_answer": generated_answer,
            "rouge1": round(scores['rouge1'].fmeasure, 4),
            "rougeL": round(scores['rougeL'].fmeasure, 4)
        })

    avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
    avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

    print(f"\nROUGE Score Results:")
    print(f"  Average ROUGE-1 : {avg_rouge1:.4f}")
    print(f"  Average ROUGE-L : {avg_rougeL:.4f}")

    return avg_rouge1, avg_rougeL, rouge_log

# Run ROUGE
avg_rouge1, avg_rougeL, rouge_log = calculate_rouge(qa_pairs)


  Preparing metadata (setup.py) ... done
Calculating ROUGE scores...

ROUGE Score Results:
  Average ROUGE-1 : 0.3017
  Average ROUGE-L : 0.2436


In [14]:
def calculate_hit_rate(qa_pairs, k=5):
    """
    Hit Rate @ K: Was the correct URL found in the top K results?
    Hit Rate = (Number of hits) / (Total questions)
    """
    hits = 0
    hit_log = []

    print(f"Calculating Hit Rate @ {k}...")
    for qa in qa_pairs:
        question = qa['question']
        correct_url = qa['source_url']

        # Run hybrid search
        retrieved = hybrid_search(question, top_k=k)

        # Check if correct URL is in top K
        retrieved_urls = [chunk['metadata']['url'] for chunk, _ in retrieved]
        is_hit = correct_url in retrieved_urls

        if is_hit:
            hits += 1

        hit_log.append({
            "id": qa['id'],
            "question": question,
            "correct_url": correct_url,
            "hit": is_hit
        })

    hit_rate = hits / len(qa_pairs)

    print(f"\nHit Rate @ {k} Results:")
    print(f"  Total Questions : {len(qa_pairs)}")
    print(f"  Total Hits      : {hits}")
    print(f"  Hit Rate @ {k}  : {hit_rate:.4f}")

    return hit_rate, hit_log

# Run Hit Rate
hit_rate, hit_log = calculate_hit_rate(qa_pairs, k=5)


Calculating Hit Rate @ 5...

Hit Rate @ 5 Results:
  Total Questions : 72
  Total Hits      : 72
  Hit Rate @ 5  : 1.0000


In [15]:
import pandas as pd

# Combine all metric logs into one results table
results_table = []
for i in range(len(qa_pairs)):
    results_table.append({
        "Question ID"     : qa_pairs[i]['id'],
        "Question"        : qa_pairs[i]['question'],
        "Ground Truth"    : qa_pairs[i]['ground_truth'],
        "Generated Answer": rouge_log[i]['generated_answer'],
        "MRR Score"       : mrr_log[i]['reciprocal_rank'],
        "ROUGE-1"         : rouge_log[i]['rouge1'],
        "ROUGE-L"         : rouge_log[i]['rougeL'],
        "Hit@5"           : hit_log[i]['hit'],
        "Source URL"      : qa_pairs[i]['source_url']
    })

df_results = pd.DataFrame(results_table)

# Print summary
print("=" * 60)
print("         FINAL EVALUATION SUMMARY")
print("=" * 60)
print(f"  Total Questions : {len(qa_pairs)}")
print(f"  MRR Score       : {mrr_score:.4f}")
print(f"  ROUGE-1         : {avg_rouge1:.4f}")
print(f"  ROUGE-L         : {avg_rougeL:.4f}")
print(f"  Hit Rate @ 5    : {hit_rate:.4f}")
print("=" * 60)

# Save to CSV
df_results.to_csv("evaluation_results.csv", index=False)
print("\nResults saved to evaluation_results.csv")

# Display first 5 rows
df_results.head()


         FINAL EVALUATION SUMMARY
  Total Questions : 72
  MRR Score       : 0.9861
  ROUGE-1         : 0.3017
  ROUGE-L         : 0.2436
  Hit Rate @ 5    : 1.0000

Results saved to evaluation_results.csv


,Question ID,Question,Ground Truth,Generated Answer,MRR Score,ROUGE-1,ROUGE-L,Hit@5,Source URL
0,1,Who recorded a cover of Charlie Rich's song at...,Diffie recorded a cover of Charlie Rich's song...,"At the end of 1998, Joe Diffie recorded a cove...",1.0,0.4909,0.4000,True,https://en.wikipedia.org/wiki/Joe_Diffie
1,2,Who attempted to set a new world record for lo...,Jeff Cassell attempted to set a new world reco...,The individual who attempted to set a new worl...,1.0,0.2468,0.2338,True,https://en.wikipedia.org/wiki/Scott_Cassell
2,3,Who led a delegation of Pakistani journalists ...,The text states that the person in question re...,"According to [Source 1], Mian Hayaud Din relin...",1.0,0.4670,0.3756,True,https://en.wikipedia.org/wiki/Mian_Hayaud_Din
3,4,"How did Thomas Rhett, Luke Laird, and Barry De...","Thomas Rhett, Luke Laird, and Barry Dean co-wr...","Thomas Rhett, Luke Laird, and Barry Dean were ...",1.0,0.3805,0.2634,True,https://en.wikipedia.org/wiki/Joe_Diffie
4,5,Who did Brad Seely work for as a special teams...,Brad Seely worked for coaches Pete Carroll and...,Before joining the New England Patriots as the...,1.0,0.2482,0.1752,True,https://en.wikipedia.org/wiki/Brad_Seely


In [16]:
!pip install -q gradio


In [17]:
import pandas as pd

# Build a DataFrame of all articles
articles_df = pd.DataFrame([
    {
        "No."  : i + 1,
        "Title": doc['title'],
        "URL"  : doc['url'],
        "Word Count": len(doc['text'].split())
    }
    for i, doc in enumerate(total_corpus)
])

# Print Summary
print("=" * 60)
print(f"  TOTAL ARTICLES IN CORPUS: {len(total_corpus)}")
print(f"  Fixed Articles          : 200")
print(f"  Random Articles         : 300")
print("=" * 60)

# Display Full Table
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 60)
print(articles_df.to_string(index=False))


  TOTAL ARTICLES IN CORPUS: 500
  Fixed Articles          : 200
  Random Articles         : 300
 No.                                                                                                                                                                                                   Title                                                                                                                                                                                                                                   URL  Word Count
   1                                                                                                                                                                               34th Annual Grammy Awards                                                                                                                                                                               https://en.wikipedia.org/wiki/34th_Annual_Grammy_Awards        1469
   2      

In [18]:
import gradio as gr
import pandas as pd
import time

# Prepare article list
article_list = [
    {
        "No."       : i + 1,
        "Title"     : doc['title'],
        "URL"       : doc['url'],
        "Word Count": len(doc['text'].split())
    }
    for i, doc in enumerate(total_corpus)
]
articles_df = pd.DataFrame(article_list)

# --- Article Search Function ---
def search_articles(keyword):
    """Filter articles by keyword in title."""
    if not keyword.strip():
        return articles_df

    filtered = articles_df[
        articles_df['Title'].str.contains(keyword, case=False, na=False)
    ]

    if filtered.empty:
        return pd.DataFrame([{"Message": f"No articles found containing '{keyword}'"}])

    return filtered

# --- Q&A Function ---
def answer_question(question):
    """Run the RAG pipeline and return answer + sources."""
    if not question.strip():
        return "⚠️ Please enter a question.", ""

    start_time = time.time()

    # Step 1: Retrieve relevant chunks
    retrieved_results = hybrid_search(question, top_k=10, top_n=5)
    context_chunks = [res[0] for res in retrieved_results]

    # Step 2: Generate answer
    answer = generate_answer(question, context_chunks)

    # Step 3: Build sources text
    seen_urls = []
    sources_text = "📚 Sources Used:\n\n"
    for chunk, score in retrieved_results:
        url   = chunk['metadata']['url']
        title = chunk['metadata']['title']
        if url not in seen_urls:
            seen_urls.append(url)
            sources_text += f"{len(seen_urls)}. {title}\n"
            sources_text += f"   🔗 {url}\n"
            sources_text += f"   📝 {chunk['text'][:150]}...\n\n"

    elapsed = time.time() - start_time
    sources_text += f"⏱️ Response Time: {elapsed:.2f} seconds"

    return answer, sources_text

# --- Build Combined Interface ---
with gr.Blocks(theme=gr.themes.Soft()) as app:

    # Header
    gr.Markdown("""
    # 🔍 Wikipedia RAG System
    ### Powered by Hybrid Retrieval (Dense + Sparse + RRF) and Flan-T5
    """)

    # Two Tabs
    with gr.Tabs():

        # ── Tab 1: Q&A ──────────────────────────────────────────
        with gr.TabItem("💬 Ask a Question"):

            gr.Markdown("### Type your question below based on the Wikipedia articles in the corpus.")

            with gr.Row():
                question_input = gr.Textbox(
                    label="Your Question",
                    placeholder="e.g. What is machine learning?",
                    lines=2,
                    scale=4
                )
                ask_btn = gr.Button(
                    "🔎 Ask",
                    variant="primary",
                    scale=1
                )

            with gr.Row():
                with gr.Column():
                    answer_output = gr.Textbox(
                        label="💬 Generated Answer",
                        lines=6,
                        interactive=False
                    )
                with gr.Column():
                    sources_output = gr.Textbox(
                        label="📚 Sources",
                        lines=6,
                        interactive=False
                    )

            # Example Questions
            gr.Examples(
                examples=[
                    ["What is the theory of relativity?"],
                    ["Who invented the telephone?"],
                    ["What is the capital of France?"],
                    ["How does photosynthesis work?"],
                    ["What caused World War 1?"],
                    ["Who was Albert Einstein?"],
                    ["What is artificial intelligence?"],
                    ["How does the human brain work?"]
                ],
                inputs=question_input,
                label=" Example Questions (Click any to try)"
            )

            # Connect Ask Button
            ask_btn.click(
                fn=answer_question,
                inputs=question_input,
                outputs=[answer_output, sources_output]
            )

            # Allow pressing Enter
            question_input.submit(
                fn=answer_question,
                inputs=question_input,
                outputs=[answer_output, sources_output]
            )

        # ── Tab 2: Article Browser ───────────────────────────────
        with gr.TabItem("📚 Browse Articles"):

            gr.Markdown("### Search and browse all Wikipedia articles in the corpus.")

            # Corpus Stats
            with gr.Row():
                gr.Markdown(f"""
                | Metric | Value |
                |--------|-------|
                | 📰 Total Articles | {len(total_corpus)} |
                | 📌 Fixed Articles | 200 |
                | 🎲 Random Articles | 300 |
                | 🧩 Total Chunks | {len(all_chunks)} |
                """)

            # Search Row
            with gr.Row():
                search_box = gr.Textbox(
                    label="🔍 Search Articles by Title",
                    placeholder="e.g. Science, History, War, Einstein...",
                    scale=4
                )
                search_btn = gr.Button("Search", variant="primary", scale=1)
                clear_btn  = gr.Button("Show All", scale=1)

            # Results Table
            results_table = gr.Dataframe(
                value=articles_df,
                label=f"Articles in Corpus ({len(total_corpus)} total)",
                interactive=False,
                wrap=True
            )

            # Tip
            gr.Markdown("""
            > 💡 **Tip:** Find an article here, then switch to the
            **💬 Ask a Question** tab to ask about it!
            """)

            # Connect Buttons
            search_btn.click(
                fn=search_articles,
                inputs=search_box,
                outputs=results_table
            )
            search_box.submit(
                fn=search_articles,
                inputs=search_box,
                outputs=results_table
            )
            clear_btn.click(
                fn=lambda: articles_df,
                inputs=None,
                outputs=results_table
            )

# Launch
app.launch(share=True, debug=True)


/tmp/ipykernel_2173/3225351283.py:65: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9f069e75dd6cc496c5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9f069e75dd6cc496c5.gradio.live


In [19]:
# ── Run this in your Colab ────────────────────────────────────
import pickle
import json
import faiss
import os
from google.colab import files
import time

print("Saving all files for HuggingFace deployment...")
print("=" * 50)

# ── 1. Save Corpus ────────────────────────────────────────────
with open("corpus.pkl", "wb") as f:
    pickle.dump(total_corpus, f)
print(f"✅ corpus.pkl → {len(total_corpus)} articles")

# ── 2. Save Chunks ────────────────────────────────────────────
with open("chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)
print(f"✅ chunks.pkl → {len(all_chunks)} chunks")

# ── 3. Save FAISS Index ───────────────────────────────────────
faiss.write_index(index, "faiss_index.bin")
print(f"✅ faiss_index.bin → {index.ntotal} vectors")

# ── 4. Save BM25 ─────────────────────────────────────────────
with open("bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)
print(f"✅ bm25_index.pkl saved")

# ── 5. Save Articles JSON ─────────────────────────────────────
article_list = [
    {
        "No."       : i + 1,
        "Title"     : doc["title"],
        "URL"       : doc["url"],
        "Word Count": len(doc["text"].split())
    }
    for i, doc in enumerate(total_corpus)
]
with open("articles_data.json", "w") as f:
    json.dump(article_list, f, indent=2)
print(f"✅ articles_data.json saved")

# ── 6. Check Eval Files ───────────────────────────────────────
for fname in ["evaluation_results.csv", "qa_pairs.json"]:
    if os.path.exists(fname):
        print(f"✅ {fname} found")
    else:
        print(f"⚠️  {fname} not found - run evaluation first!")

# ── 7. Show File Sizes ────────────────────────────────────────
print("\n📦 File Sizes:")
print("=" * 50)
all_files = [
    "corpus.pkl",
    "chunks.pkl",
    "faiss_index.bin",
    "bm25_index.pkl",
    "articles_data.json",
    "evaluation_results.csv",
    "qa_pairs.json"
]
total_mb = 0
for fname in all_files:
    if os.path.exists(fname):
        mb = os.path.getsize(fname) / (1024 * 1024)
        total_mb += mb
        print(f"  {fname:<30} {mb:.2f} MB")

print(f"\n  {'TOTAL':<30} {total_mb:.2f} MB")
print("=" * 50)

# ── 8. Download All ───────────────────────────────────────────
print("\nStarting downloads...")
print("⚠️ Allow each download in your browser!")
print("=" * 50)

for fname in all_files:
    if os.path.exists(fname):
        print(f"Downloading {fname}...")
        files.download(fname)
        time.sleep(2)

print("\n✅ All downloads complete!")


Saving all files for HuggingFace deployment...
✅ corpus.pkl → 500 articles
✅ chunks.pkl → 11121 chunks
✅ faiss_index.bin → 11121 vectors
✅ bm25_index.pkl saved
✅ articles_data.json saved
✅ evaluation_results.csv found
✅ qa_pairs.json found

📦 File Sizes:
  corpus.pkl                     2.77 MB
  chunks.pkl                     3.51 MB
  faiss_index.bin                16.29 MB
  bm25_index.pkl                 4.72 MB
  articles_data.json             0.07 MB
  evaluation_results.csv         0.09 MB
  qa_pairs.json                  0.04 MB

  TOTAL                          27.49 MB

Starting downloads...
⚠️ Allow each download in your browser!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All downloads complete!
